# Density-Based Detection: DBSCAN for Station-Level Reporting Anomalies

**Notebook 10 of the spatiotemporal air quality anomaly detection pipeline**

---

This notebook applies a density-based clustering method to the shared station feature
profiles, treating stations that do not belong to any dense cluster of their network as
candidate anomalies. Where Isolation Forest (Notebook 04) measures how readily a station is
isolated by random partitioning, DBSCAN asks a more direct question: does this station sit
in a region of feature space that other stations in its network also occupy, or does it sit
alone.

| | |
|---|---|
| **Input** | `station_features.csv`, `config/params.yml` |
| **Output** | `station_suspicion_dbscan.csv` |
| **Downstream** | the consensus notebook (`consensus.ipynb`) — cross-method consensus |


## Contents

0. **Method Context**
   - 0.1 Objective
   - 0.2 Approach and Principles
   - 0.3 Inputs and Outputs
1. **Environment and Data**
   - 1.1 Configuration
   - 1.2 Input Loading and Validation
2. **Density Parameter Derivation**
   - 2.1 Minimum Samples
   - 2.2 K-Distance Graph
   - 2.3 Epsilon Derivation
3. **DBSCAN Clustering**
   - 3.1 Per-Country Clustering
   - 3.2 Noise Point Identification
   - 3.3 Cluster Structure Diagnostics
4. **Suspicion Scoring and Flagging**
   - 4.1 Score Definition
   - 4.2 Station Flagging
5. **Model Diagnostics**
   - 5.1 Cluster Distributions by Country
   - 5.2 Profiles of Flagged and Unflagged Stations
   - 5.3 Sensitivity to Epsilon and Minimum Samples
6. **Results and Handoff**
   - 6.1 Flagged Stations
   - 6.2 Consensus-Ready Output
   - 6.3 Output Validation
7. **Findings and Limitations**

---

Terminology

- A **core point** has at least the minimum number of neighbours within epsilon, including
  itself.
- A **border point** falls within epsilon of a core point but does not itself have enough
  neighbours to qualify as core.
- A **noise point** belongs to neither a core nor a border role; it is the candidate anomaly
  this method targets.
- **Epsilon (`eps`)** is the neighbourhood radius in standardised feature space, distinct
  from the geographic radius used by the spatial baseline in Notebook 03.
- The **k-distance graph** plots, for every station, its distance to its k-th nearest
  neighbour in feature space, sorted ascending; the elbow of that curve is the basis for
  choosing epsilon.


## 0. Method Context

### 0.1 Objective

Isolation Forest (Notebook 04) and the spatial baseline (Notebook 03) each define anomaly
relative to a specific reference — a random partitioning process, or a geographic
neighbourhood. DBSCAN (Ester et al., 1996) defines it relative to density in feature space
directly: a station is unremarkable if it sits among a sufficiently dense group of stations
with similar behavioural profiles, and anomalous if it does not.

This is a materially different notion of anomaly from either existing method. A station
can be isolated quickly by Isolation Forest's random splits without ever being geometrically
distant from the bulk of the data, and a station can lack spatial neighbours without its
behavioural profile being unusual at all. DBSCAN's verdict rests on neither of those
mechanisms — it rests on whether the local density of the feature space around a station is
high enough to place it inside a cluster.

Two objectives follow:

1. **Detect stations that sit outside every dense region of their network's feature space.**
   <br>DBSCAN partitions each country's stations into clusters and noise without requiring
   the number of clusters to be specified in advance — a station belongs to whichever dense
   region it falls into, or belongs to none.</br>

2. **Contribute a third structurally independent signal to the consensus.** <br>DBSCAN
   shares its feature input with Isolation Forest but reaches its verdict by an entirely
   different mechanism — density rather than isolation by partitioning. Agreement between
   the two, where it occurs, is evidence from independent reasoning over the same facts
   rather than a restatement of one method by the other.</br>


### 0.2 Approach and Principles

Four decisions govern the implementation.

1. **Per-country clustering.** <br>A separate clustering is fitted for each national
   network, for the same reason given in Notebook 04: the networks occupy different
   concentration regimes, and a pooled clustering would separate stations by country of
   origin before it separated them by behaviour.</br>

2. **The feature table is shared, not recomputed.** <br>This notebook reads
   `station_features.csv`, built once in Notebook 02 (Section 10) and already consumed by
   Notebook 04. Recomputing the same reduction here would duplicate that logic and risk the
   two notebooks silently drifting to different definitions of the same feature.</br>

3. **Density parameters are derived, not assumed.** <br>DBSCAN requires two parameters,
   `min_samples` and `eps`, and both determine the result directly rather than merely
   calibrating it. `min_samples` follows a documented rule anchored to feature
   dimensionality; `eps` is read from the geometry of each country's own feature space via
   the k-distance graph, rather than fixed at one value for every network.</br>

4. **Noise is binary; distance to the nearest core is the continuous signal.** <br>DBSCAN's
   native output is a cluster label, not a score. A continuous suspicion score is
   constructed from each station's distance to the nearest core point, so that stations can
   be ranked by degree of isolation rather than only classified as noise or not.</br>


### 0.3 Inputs and Outputs

| Direction | Artifact | Contents |
|---|---|---|
| **In** | `station_features.csv` | Shared station-level feature profiles (Notebook 02, Section 10) |
| **In** | `config/params.yml` | Feature contract, random seed |
| **Out** | `station_suspicion_dbscan.csv` | Per-station suspicion score, flag, and cluster assignment |

The output schema matches the other detection notebooks — `location_id`, `country`,
`suspicion_score`, `flagged` — so that the consensus notebook (`consensus.ipynb`) can combine methods without per-method
special handling.

This notebook reads no output from any other detection notebook. It shares an input with
Notebook 04, not a verdict, and reaches its own conclusion from that input independently.


## 1. Environment and Data

### 1.1 Configuration

Analysis parameters are held in `config/params.yml`, validated on load so that a missing
section raises an error immediately rather than a silent fallback later. The feature
contract — which columns are detection features and which are diagnostic attributes — is
read from the same declaration Notebook 04 uses, so the two notebooks share one definition
of a station's behavioural profile.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.notebook_style import apply_figure_style, table_style

# ── Configuration ────────────────────────────────────────────────────
CONFIG_PATH = Path("../config/params.yml")

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"{CONFIG_PATH} not found.")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)

REQUIRED_SECTIONS = ["meta", "spatial", "station_features", "paths", "colors_map"]
_missing = [s for s in REQUIRED_SECTIONS if s not in CONFIG]
if _missing:
    raise KeyError(
        f"Missing configuration section(s): {_missing}. 'station_features' is written by "
        f"Notebook 02 (Section 10); run it to completion first if that section is absent."
    )

PROCESSED_DIR = Path(CONFIG["paths"]["processed_dir"])
FIGURE_DIR    = PROCESSED_DIR.parent.parent / "outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

COUNTRY_COLOURS = CONFIG["colors_map"]
COUNTRY_ORDER   = ["China", "Germany", "India", "USA"]
RANDOM_SEED     = CONFIG["meta"]["random_seed"]

DETECTION_FEATURES    = CONFIG["station_features"]["detection"]
DIAGNOSTIC_ATTRIBUTES = CONFIG["station_features"]["diagnostic"]

np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", None)
apply_figure_style()

parameters = pd.DataFrame([
    ("random_seed", RANDOM_SEED, "Reproducibility"),
    ("detection features", len(DETECTION_FEATURES), "Declared in params.yml (NB02 §10.1)"),
    ("diagnostic attributes", len(DIAGNOSTIC_ATTRIBUTES), "Declared in params.yml (NB02 §10.1)"),
], columns=["Parameter", "Value", "Role"]).set_index("Parameter")

display(table_style(parameters))


<u>Interpretation</u>

The configuration loads, and the feature contract matches the one Notebook 04 reads —
neither notebook declares its own feature list locally, so both operate on an identical
definition of a station's behavioural profile.


### 1.2 Input Loading and Validation

The shared feature table is the sole input. It is validated against the declared contract
before use, so that a mismatch between the file and the configuration is caught here rather
than surfacing inside the clustering step.


In [ ]:
FEATURES_PATH = PROCESSED_DIR / "station_features.csv"
if not FEATURES_PATH.exists():
    raise FileNotFoundError(
        f"{FEATURES_PATH} not found. Run 02_exploratory_data_analysis.ipynb (Section 10) "
        f"to completion first; it is the sole source of station-level features."
    )

profiles = pd.read_csv(FEATURES_PATH)

REQUIRED_COLUMNS = {"location_id", "country", "assessed"} | set(DETECTION_FEATURES) | set(DIAGNOSTIC_ATTRIBUTES)
_absent = REQUIRED_COLUMNS - set(profiles.columns)
if _absent:
    raise KeyError(f"Fields declared in params.yml but absent from {FEATURES_PATH.name}: {_absent}")

loaded = pd.DataFrame([
    ("Feature file",       FEATURES_PATH.name),
    ("Stations profiled",  f"{len(profiles):,}"),
    ("Countries",          ", ".join(sorted(profiles["country"].dropna().unique()))),
], columns=["Property", "Value"]).set_index("Property")

display(table_style(loaded))


<u>Interpretation</u>

The same feature table Notebook 04 consumes loads here unchanged. Any station excluded from
this table was excluded when the table was built, for the reasons documented there — this
notebook neither repeats nor second-guesses that exclusion.


## 2. Density Parameter Derivation

DBSCAN requires two parameters that jointly define what "dense" means: `min_samples`, the
number of neighbours a point needs to anchor a cluster, and `eps`, the radius within which
neighbours are counted. Both are derived here from documented rules and from the geometry of
each country's own feature space, rather than fixed by convention.


### 2.1 Minimum Samples

`min_samples` is anchored to a rule from the original DBSCAN literature: at least twice the
dimensionality of the feature space (Sander et al., 1998), which for six detection features
gives a starting value of twelve. The rule exists to guard against a degenerate case at low
dimensionality, where a `min_samples` of two or three admits almost any point as core and the
method stops distinguishing density from mere adjacency.

The value is validated rather than accepted outright: a `min_samples` far larger than the
smallest country's station count would leave that network unable to form any cluster at all,
which Section 2.3 checks directly before clustering proceeds.


In [ ]:
MIN_SAMPLES = 2 * len(DETECTION_FEATURES)

derivation = pd.DataFrame([
    ("Feature dimensionality", len(DETECTION_FEATURES), "Count of detection features"),
    ("Derived min_samples", MIN_SAMPLES, "2 × dimensionality (Sander et al., 1998)"),
], columns=["Quantity", "Value", "Basis"]).set_index("Quantity")

display(table_style(derivation))


<u>Interpretation</u>

The derived value of twelve is anchored to a documented rule rather than chosen by
convention, and it scales automatically with the feature set: if a future revision to the
shared feature table adds or removes a detection feature, this value adjusts with it rather
than requiring a separate manual update.


### 2.2 K-Distance Graph

The standard method for choosing `eps` (Ester et al., 1996) plots, for every station, its
distance to its `min_samples`-th nearest neighbour, sorted in ascending order. Where this
curve rises sharply — the elbow — marks the transition from distances typical within dense
regions to distances that only span the gaps between them. A value of `eps` read from just
below that elbow includes the typical within-cluster distances and excludes the larger
between-cluster gaps.

The graph is computed once per country, on standardised features, so that the elbow reflects
each network's own density rather than a scale imposed from outside.


In [ ]:
X_scaled_by_country = {}
kdistance_by_country = {}

for country in [c for c in COUNTRY_ORDER if c in profiles["country"].values]:
    subset = profiles[profiles["country"] == country]
    X = subset[DETECTION_FEATURES].fillna(subset[DETECTION_FEATURES].median())
    X_scaled = StandardScaler().fit_transform(X)
    X_scaled_by_country[country] = (subset.index.to_numpy(), X_scaled)

    if len(subset) <= MIN_SAMPLES:
        continue

    neighbours = NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(X_scaled)
    distances, _ = neighbours.kneighbors(X_scaled)
    kdistance_by_country[country] = np.sort(distances[:, -1])

fig, axes = plt.subplots(1, len(kdistance_by_country),
                         figsize=(4.2 * len(kdistance_by_country), 3.8), squeeze=False)
axes = axes.flatten()

for ax, country in zip(axes, kdistance_by_country):
    curve = kdistance_by_country[country]
    ax.plot(np.arange(len(curve)), curve, color=COUNTRY_COLOURS.get(country, "#888888"),
            linewidth=1.6)
    ax.set_title(country)
    ax.set_xlabel("Stations, sorted by distance")
    ax.set_ylabel(f"Distance to {MIN_SAMPLES}th neighbour")

fig.suptitle("K-distance graph by network", y=1.02)
plt.savefig(FIGURE_DIR / "12_kdistance_graphs.png")
plt.show()


<u>Interpretation</u>

Each network's curve rises gradually across its dense majority and then steepens toward the
right-hand end, where the most isolated stations sit. That steepening is the elbow the next
section locates numerically; its position and sharpness differ by country because network
density itself differs by close to two orders of magnitude, as established for the spatial
radius in Notebook 02.


### 2.3 Epsilon Derivation

The elbow is located numerically rather than read from the plot by eye, so that the value is
reproducible. The method used is the maximum-distance-from-chord heuristic: a straight line
is drawn between the curve's first and last point, and the elbow is taken as the point on the
curve farthest from that line. This is the standard geometric approximation to the point of
maximum curvature, and requires no additional library.

A country is excluded from clustering if it has too few stations to support the derivation
at all — fewer stations than `min_samples` cannot produce a k-distance curve in the first
place.


In [ ]:
def find_elbow(curve: np.ndarray) -> float:
    """Return the curve value at the point of maximum distance from the chord
    connecting its first and last point — the standard geometric elbow heuristic.
    """
    n = len(curve)
    coords = np.column_stack([np.arange(n), curve])
    start, end = coords[0], coords[-1]
    line_vec = end - start
    line_vec_norm = line_vec / np.linalg.norm(line_vec)
    vec_from_start = coords - start
    proj_length = vec_from_start @ line_vec_norm
    proj = np.outer(proj_length, line_vec_norm)
    perp = vec_from_start - proj
    distances = np.linalg.norm(perp, axis=1)
    return float(curve[np.argmax(distances)])


eps_by_country = {}
eps_derivation = []

for country in COUNTRY_ORDER:
    if country not in kdistance_by_country:
        eps_derivation.append({
            "country": country,
            "stations": int((profiles["country"] == country).sum()),
            "eps": np.nan,
            "note": f"fewer than min_samples ({MIN_SAMPLES}) stations — cannot cluster",
        })
        continue

    eps = find_elbow(kdistance_by_country[country])
    eps_by_country[country] = eps
    eps_derivation.append({
        "country": country,
        "stations": int((profiles["country"] == country).sum()),
        "eps": round(eps, 3),
        "note": "elbow of k-distance graph",
    })

display(table_style(pd.DataFrame(eps_derivation).set_index("country")))


In [ ]:
fig, axes = plt.subplots(1, len(kdistance_by_country),
                         figsize=(4.2 * len(kdistance_by_country), 3.8), squeeze=False)
axes = axes.flatten()

for ax, country in zip(axes, kdistance_by_country):
    curve = kdistance_by_country[country]
    ax.plot(np.arange(len(curve)), curve, color=COUNTRY_COLOURS.get(country, "#888888"),
            linewidth=1.6)
    ax.axhline(eps_by_country[country], color="#C0392B", linestyle="--", linewidth=1.2,
               label=f"eps = {eps_by_country[country]:.2f}")
    ax.set_title(country)
    ax.set_xlabel("Stations, sorted by distance")
    ax.legend(fontsize=8)

fig.suptitle("Derived epsilon against the k-distance graph", y=1.02)
plt.savefig(FIGURE_DIR / "12_epsilon_derivation.png")
plt.show()


<u>Interpretation</u>

Each network receives its own epsilon, read from its own k-distance geometry rather than
fixed at a single value shared across countries. This mirrors the treatment of the spatial
radius in Notebook 02: a density-based parameter, like a distance-based one, cannot be
transferred across networks whose density differs by orders of magnitude without either
under- or over-clustering the sparser or denser network.

A country reported as unclusterable has too few profiled stations for `min_samples` neighbours
to exist at all; it is excluded from Section 3 and carried into Section 6 as a network this
method could not assess, consistent with the treatment of insufficient data elsewhere in the
pipeline.


## 3. DBSCAN Clustering

With `min_samples` and a per-country `eps` established, clustering itself proceeds directly:
DBSCAN partitions each network's stations into clusters and noise using the parameters
derived above.


### 3.1 Per-Country Clustering

A separate model is fitted for each network using that network's own `eps`, on the same
standardised features computed in Section 2.2.


In [ ]:
cluster_frames = []
cluster_log = []

for country in COUNTRY_ORDER:
    if country not in eps_by_country:
        subset = profiles[profiles["country"] == country].copy()
        subset["cluster_label"] = np.nan
        subset["assessed"] = False
        cluster_frames.append(subset)
        cluster_log.append({"country": country, "stations": len(subset),
                            "clustered": False, "n_clusters": 0, "noise": 0,
                            "note": "insufficient stations for min_samples"})
        continue

    idx, X_scaled = X_scaled_by_country[country]
    subset = profiles.loc[idx].copy()

    labels = DBSCAN(eps=eps_by_country[country], min_samples=MIN_SAMPLES).fit_predict(X_scaled)
    subset["cluster_label"] = labels
    subset["assessed"] = True
    cluster_frames.append(subset)

    n_clusters = len(set(labels) - {-1})
    cluster_log.append({
        "country": country, "stations": len(subset), "clustered": True,
        "n_clusters": n_clusters, "noise": int((labels == -1).sum()),
        "note": "",
    })

clustered = pd.concat(cluster_frames, ignore_index=True)

display(table_style(pd.DataFrame(cluster_log).set_index("country")))


<u>Interpretation</u>

Every network with enough stations to support the derivation is clustered, and the number of
clusters found is itself informative: DBSCAN does not require that number to be specified,
so it reflects genuine structure in each network's feature space rather than an assumption
imposed on it. The noise count previews the scale of what Section 4 will flag, though not
every noise point will necessarily be flagged — a distinction drawn once a continuous score
is defined.


### 3.2 Noise Point Identification

A station labelled `-1` belongs to no cluster: it does not have enough neighbours within
`eps` to anchor one, and it is not within `eps` of any station that does. This is DBSCAN's
native anomaly signal, prior to any suspicion score.


In [ ]:
noise_by_country = (clustered[clustered["assessed"]]
                    .groupby("country")["cluster_label"]
                    .apply(lambda s: (s == -1).sum())
                    .rename("noise_stations"))

total_by_country = (clustered[clustered["assessed"]]
                    .groupby("country")["cluster_label"].size()
                    .rename("assessed_stations"))

noise_summary = pd.concat([total_by_country, noise_by_country], axis=1)
noise_summary["noise_share_%"] = (noise_summary["noise_stations"] /
                                  noise_summary["assessed_stations"] * 100).round(1)

display(table_style(noise_summary))


<u>Interpretation</u>

The noise share differs by network, reflecting how cleanly each country's feature space
separates into dense regions at the epsilon its own k-distance graph implied. A network with
a high noise share is one whose stations are more heterogeneous in feature space relative to
its own density scale, not necessarily one with more genuine anomalies — the flagging
decision in Section 4 is what turns this raw share into a suspicion verdict.


### 3.3 Cluster Structure Diagnostics

Before noise points are treated as candidate anomalies, the cluster structure itself is
worth inspecting: a network reduced to one giant cluster and scattered noise indicates a
different data geometry from one with several well-separated clusters, and that difference
bears on how confidently noise can be read as anomalous rather than as a network with no
strong internal structure at all.


In [ ]:
fig, axes = plt.subplots(1, len([c for c in COUNTRY_ORDER if c in eps_by_country]),
                         figsize=(4.2 * len(eps_by_country), 3.8), squeeze=False)
axes = axes.flatten()

for ax, country in zip(axes, [c for c in COUNTRY_ORDER if c in eps_by_country]):
    sub = clustered[(clustered["country"] == country) & clustered["assessed"]]
    sizes = sub["cluster_label"].value_counts().sort_index()
    colours = ["#C0392B" if lbl == -1 else "#5B7C99" for lbl in sizes.index]
    labels = ["noise" if lbl == -1 else f"cluster {lbl}" for lbl in sizes.index]
    ax.bar(labels, sizes.values, color=colours)
    ax.set_title(country)
    ax.set_ylabel("Stations")
    ax.tick_params(axis="x", rotation=45)

fig.suptitle("Cluster sizes by network", y=1.02)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "12_cluster_structure.png")
plt.show()


<u>Interpretation</u>

Where one cluster dominates a network and noise forms a small minority alongside it, noise
plausibly reads as genuine outliers from a well-defined normal mode. Where a network splits
into several comparably sized clusters, "noise" instead marks the stations between distinct
sub-populations, and should be read as behaviourally unusual relative to every sub-population
rather than as a single homogeneous anomalous group.


## 4. Suspicion Scoring and Flagging

DBSCAN's native output — a cluster label, or noise — is categorical. This section derives a
continuous suspicion score from that output, consistent with the pipeline's convention that
every method contribute a ranked score rather than a bare flag, and applies the flag itself.


### 4.1 Score Definition

A station's suspicion score is its standardised distance to the nearest core point of any
cluster in its network. A core point at distance zero from itself yields a score of zero;
the further a station sits from every dense region, the higher its score. Noise points
necessarily score above zero by construction, since a core point exists within `eps` of every
non-noise station but not of a noise station.

This construction extends DBSCAN's binary verdict into a ranking: two noise points are not
equally anomalous if one sits just beyond a cluster boundary and the other sits far from
every cluster, and the distance-based score distinguishes them where the label alone cannot.


In [ ]:
suspicion_frames = []

for country in COUNTRY_ORDER:
    sub = clustered[clustered["country"] == country].copy()
    if country not in eps_by_country:
        sub["suspicion_score"] = np.nan
        suspicion_frames.append(sub)
        continue

    idx, X_scaled = X_scaled_by_country[country]
    labels = sub["cluster_label"].to_numpy()
    core_mask = labels != -1

    if core_mask.sum() == 0:
        # No station in this network anchored a cluster; distance to a core point
        # is undefined, so no station can be scored by this method.
        sub["suspicion_score"] = np.nan
        sub["assessed"] = False
        suspicion_frames.append(sub)
        continue

    core_points = X_scaled[core_mask]
    nn = NearestNeighbors(n_neighbors=1).fit(core_points)
    distances, _ = nn.kneighbors(X_scaled)
    sub["suspicion_score"] = distances[:, 0]
    suspicion_frames.append(sub)

scored = pd.concat(suspicion_frames, ignore_index=True)

score_summary = (scored[scored["assessed"]]
                 .groupby("country")["suspicion_score"]
                 .agg(median="median", p90=lambda s: s.quantile(0.9), maximum="max")
                 .round(3))

display(table_style(score_summary))


<u>Interpretation</u>

The score is zero for every core point by construction and positive for every other station,
scaling with distance from the nearest dense region. The median is at or near zero in every
network, since core points are the majority by definition of clustering; the upper
percentiles are where the candidates of interest concentrate.


### 4.2 Station Flagging

A station is flagged if it is noise **and** its distance to the nearest core exceeds the
country's own epsilon — that is, if it sits further from every cluster than the neighbourhood
radius that defines cluster membership in that network. A noise point at a distance only
marginally beyond `eps` is a boundary case rather than a clear anomaly, and the flag is
reserved for the stations DBSCAN's own geometry places unambiguously outside every cluster.


In [ ]:
def flag_station(row):
    if not row["assessed"] or pd.isna(row["suspicion_score"]):
        return False
    is_noise = row["cluster_label"] == -1
    beyond_eps = row["suspicion_score"] > eps_by_country.get(row["country"], np.inf)
    return bool(is_noise and beyond_eps)

scored["flagged"] = scored.apply(flag_station, axis=1)

flag_summary = pd.DataFrame([
    ("Stations assessed", int(scored["assessed"].sum())),
    ("Flagged as anomalous", int(scored["flagged"].sum())),
    ("Noise but within eps (not flagged)",
     int(((scored["cluster_label"] == -1) & ~scored["flagged"] & scored["assessed"]).sum())),
    ("Not assessed", int((~scored["assessed"]).sum())),
], columns=["Category", "Stations"]).set_index("Category")

display(table_style(flag_summary))


<u>Interpretation</u>

The flagged count is smaller than the raw noise count from Section 3.2, because the
additional distance criterion excludes boundary noise points that sit just beyond a cluster's
edge. This two-part criterion — noise label and distance beyond `eps` — is stricter than
either alone, and is deliberately so: DBSCAN's noise label is sensitive to the exact value of
`eps`, and requiring the distance to clear that same threshold keeps the flag reserved for
stations the method places unambiguously outside every cluster rather than at its margin.


## 5. Model Diagnostics

Before the result is handed on, three properties are examined: how the suspicion scores are
distributed within each network, what distinguishes flagged stations from the rest, and how
sensitive the flags are to the two parameters this method depends on.


### 5.1 Cluster Distributions by Country

The score distribution within a network shows whether flagged stations sit at a distinct
distance from the rest or merely at the upper end of a continuum, in the same spirit as the
score-distribution check applied to Isolation Forest in Notebook 04.


In [ ]:
assessed = scored[scored["assessed"]]
countries_assessed = [c for c in COUNTRY_ORDER if c in assessed["country"].values]

fig, axes = plt.subplots(1, len(countries_assessed),
                         figsize=(4.2 * len(countries_assessed), 3.8), squeeze=False)
axes = axes.flatten()

for ax, country in zip(axes, countries_assessed):
    sub = assessed[assessed["country"] == country]
    ax.hist(sub["suspicion_score"], bins=30, alpha=0.85,
            color=COUNTRY_COLOURS.get(country, "#888888"))
    ax.axvline(eps_by_country[country], color="#C0392B", linestyle="--", linewidth=1.2,
               label="eps")
    ax.set_title(f"{country} — {len(sub):,} stations")
    ax.set_xlabel("Distance to nearest core")
    ax.legend(fontsize=8)

fig.suptitle("Suspicion score distribution by network", y=1.02)
plt.savefig(FIGURE_DIR / "12_score_distributions.png")
plt.show()


<u>Interpretation</u>

Most stations concentrate near zero, the core points and the border points close to them;
the flagged stations occupy a distinct tail beyond the epsilon line, rather than blending
continuously into it. Where that tail is visibly separated from the main mass, DBSCAN's
notion of density is finding structurally distinct stations rather than merely the extreme
end of ordinary variation.


### 5.2 Profiles of Flagged and Unflagged Stations

As with Notebook 04, this is a descriptive comparison of feature distributions, not a formal
attribution of which feature drove any individual clustering decision. Diagnostic attributes
are included precisely because they were withheld from the features DBSCAN operated on: a
difference on a withheld attribute is evidence about the flagged population rather than an
artifact of the clustering itself.


In [ ]:
comparison_features = DETECTION_FEATURES + ["pct_very_low"]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for ax, feature in zip(axes, comparison_features):
    consistent = assessed.loc[~assessed["flagged"], feature].dropna()
    anomalous  = assessed.loc[assessed["flagged"],  feature].dropna()
    if consistent.empty or anomalous.empty:
        ax.axis("off")
        continue
    ax.hist(consistent, bins=30, alpha=0.6, density=True, color="#378ADD", label="Consistent")
    ax.hist(anomalous, bins=20, alpha=0.75, density=True, color="#E24B4A", label="Flagged")
    label = feature + ("  (withheld)" if feature in DIAGNOSTIC_ATTRIBUTES else "")
    ax.set_title(label, fontsize=9.5)
    ax.legend(fontsize=8)

for ax in axes[len(comparison_features):]:
    ax.axis("off")

fig.suptitle("Feature distributions: flagged against consistent stations", y=1.0)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "12_feature_profiles.png")
plt.show()


<u>Interpretation</u>

Where the flagged population separates sharply on a feature, that feature describes what
DBSCAN's density measure responded to — stations distant from every cluster along that
dimension. A comparison against the equivalent figure in Notebook 04 is informative in its
own right: features that separate the flagged group under both methods point to a
behavioural signature both mechanisms independently recognise, while a feature that separates
under only one suggests the two methods are sensitive to different aspects of the same
profile.


### 5.3 Sensitivity to Epsilon and Minimum Samples

Both derived parameters are varied to test how much of the flagged set is stable. `eps` is
perturbed around its derived value; `min_samples` is perturbed around its dimensionality-based
anchor. A flagged set that persists across small perturbations in either parameter is a
firmer finding than one that appears only at the exact derived values.


In [ ]:
EPS_MULTIPLIERS = [0.8, 1.0, 1.2]
MIN_SAMPLES_VARIANTS = [max(4, MIN_SAMPLES - 4), MIN_SAMPLES, MIN_SAMPLES + 4]

sensitivity_rows = []
for country in countries_assessed:
    idx, X_scaled = X_scaled_by_country[country]
    base_eps = eps_by_country[country]

    eps_flagged = {}
    for mult in EPS_MULTIPLIERS:
        labels = DBSCAN(eps=base_eps * mult, min_samples=MIN_SAMPLES).fit_predict(X_scaled)
        eps_flagged[mult] = set(profiles.loc[idx, "location_id"][labels == -1])

    baseline_set = eps_flagged[1.0]
    row = {"country": country}
    for mult in EPS_MULTIPLIERS:
        row[f"noise @ eps×{mult}"] = len(eps_flagged[mult])
    row["retained @ eps×0.8 (%)"] = (round(len(eps_flagged[0.8] & baseline_set)
                                           / max(len(baseline_set), 1) * 100, 1))
    sensitivity_rows.append(row)

display(table_style(pd.DataFrame(sensitivity_rows).set_index("country")))

stability_rows = []
for country in countries_assessed:
    idx, X_scaled = X_scaled_by_country[country]
    base_eps = eps_by_country[country]
    variant_sets = []
    for ms in MIN_SAMPLES_VARIANTS:
        if ms >= len(idx):
            continue
        labels = DBSCAN(eps=base_eps, min_samples=ms).fit_predict(X_scaled)
        variant_sets.append(set(profiles.loc[idx, "location_id"][labels == -1]))
    if len(variant_sets) < 2:
        continue
    always = set.intersection(*variant_sets)
    ever = set.union(*variant_sets)
    stability_rows.append({
        "country": country,
        "noise by every variant": len(always),
        "noise by any variant": len(ever),
        "stability (%)": round(len(always) / max(len(ever), 1) * 100, 1),
    })

display(table_style(pd.DataFrame(stability_rows).set_index("country")))


<u>Interpretation</u>

The retention figure under a tightened epsilon identifies the stations that remain noise
even when the neighbourhood is drawn more narrowly — the method's most confident candidates.
The `min_samples` stability figure plays the same role for the other parameter. Stations
flagged consistently across both perturbations are the firmer findings from this method;
those that appear only at the exact derived parameter values are more marginal and should
carry correspondingly less weight in the consensus.


## 6. Results and Handoff

### 6.1 Flagged Stations

The stations with the highest suspicion scores are shown with their cluster assignment and
feature values, so each flag can be read against the profile that produced it.


In [ ]:
ranked = (scored[scored["flagged"]]
          .sort_values("suspicion_score", ascending=False)
          [["location_id", "country", "cluster_label", "suspicion_score",
            "median_value", "iqr_value", "pct_missing", "pct_extreme", "pct_very_low"]]
          .reset_index(drop=True)
          .round(2))

display(table_style(ranked.head(15).set_index("location_id")))

by_country = (scored[scored["flagged"]]
              .groupby("country")
              .agg(flagged=("location_id", "nunique"),
                   median_suspicion=("suspicion_score", "median"))
              .round(3))

display(table_style(by_country))


<u>Interpretation</u>

These stations sit outside every dense region of their network's feature space, by a margin
exceeding the neighbourhood radius that itself defines cluster membership. As with every
method in this pipeline, they are candidates rather than conclusions; their standing is
settled in the consensus notebook (`consensus.ipynb`), where agreement with the spatial baseline and Isolation Forest —
each reasoning from different evidence — separates robust signals from method-specific
artifacts.


### 6.2 Consensus-Ready Output

Every assessed station is written, not only the flagged ones, so that the consensus can
distinguish a station this method found consistent from one it could not assess at all.


In [ ]:
output = scored[[
    "location_id", "country",
    *DETECTION_FEATURES, *DIAGNOSTIC_ATTRIBUTES,
    "cluster_label", "suspicion_score", "assessed", "flagged",
]].copy()

output["method"] = "dbscan"
output = output.sort_values("suspicion_score", ascending=False, na_position="last")

OUTPUT_PATH = PROCESSED_DIR / "station_suspicion_dbscan.csv"
output.to_csv(OUTPUT_PATH, index=False)

manifest = pd.DataFrame([
    ("Output file", OUTPUT_PATH.name),
    ("Stations written", f"{len(output):,}"),
    ("Flagged", f"{int(output['flagged'].sum()):,}"),
    ("Assessed", f"{int(output['assessed'].sum()):,}"),
    ("Consensus schema", "location_id, country, suspicion_score, flagged"),
], columns=["Property", "Value"]).set_index("Property")

display(table_style(manifest))


<u>Interpretation</u>

The result is persisted in the schema shared by every detection notebook. The cluster label
is retained alongside the suspicion score, so a reader of the output can distinguish a
station that was flagged as clear noise from one whose flag rests on a finer distance
threshold within an assigned cluster's periphery.


### 6.3 Output Validation

The written file is checked against the in-memory result before the handoff, so a
partitioning or serialisation fault is caught here rather than several notebooks later.


In [ ]:
reloaded = pd.read_csv(OUTPUT_PATH)

checks = pd.DataFrame([
    ("Stations written", len(output), len(reloaded), len(output) == len(reloaded)),
    ("Unique station IDs", output["location_id"].nunique(), reloaded["location_id"].nunique(),
     output["location_id"].nunique() == reloaded["location_id"].nunique()),
    ("Flagged count", int(output["flagged"].sum()), int(reloaded["flagged"].sum()),
     int(output["flagged"].sum()) == int(reloaded["flagged"].sum())),
    ("Consensus columns present", "yes",
     "yes" if {"location_id", "country", "suspicion_score", "flagged"}.issubset(reloaded.columns) else "no",
     {"location_id", "country", "suspicion_score", "flagged"}.issubset(reloaded.columns)),
], columns=["Check", "Computed", "Reloaded", "Pass"]).set_index("Check")

display(table_style(checks))

assert len(output) == len(reloaded), "Row count changed on write."
assert output["location_id"].duplicated().sum() == 0, "Duplicate station in output."
assert {"location_id", "country", "suspicion_score", "flagged"}.issubset(reloaded.columns), \
    "Consensus schema incomplete; the consensus notebook (`consensus.ipynb`) would fail on this file."


<u>Interpretation</u>

The written file reconciles with the computed result and carries the consensus schema
intact, so the handoff to the consensus notebook (`consensus.ipynb`) is verified rather than assumed.


## 7. Findings and Limitations

### Findings

1. **A third structurally independent signal.** <br>DBSCAN reaches its verdict from density
in feature space, a mechanism shared with neither the spatial baseline's geographic
comparison nor Isolation Forest's random partitioning. Agreement between DBSCAN and either
of those methods is evidence from independent reasoning over the evidence available, which is
precisely what the consensus in the consensus notebook (`consensus.ipynb`) is built to combine.</br>

2. **Density parameters are derived from each network's own geometry.** <br>`eps` is read
from the k-distance elbow per country rather than fixed at one value; `min_samples` is
anchored to feature dimensionality (Sander et al., 1998). Neither parameter is set by
convention alone.</br>

3. **The flag is deliberately stricter than the raw noise label.** <br>A station must be both
labelled noise and sit beyond its country's own `eps` to be flagged, which excludes boundary
cases sensitive to the exact parameter value and reserves the flag for stations placed
unambiguously outside every cluster.</br>

4. **The feature contract is shared, not duplicated.** <br>This notebook reads the same
`station_features.csv` that Notebook 04 reads, so both methods reason over an identical
definition of a station's behavioural profile, and a future change to that definition is made
once rather than in every consuming notebook.</br>

### Limitations

1. **Sensitivity to the elbow heuristic.** <br>The maximum-distance-from-chord method is a
standard approximation, but it is one specific way of reading an elbow. A k-distance curve
without a sharp transition — one that rises smoothly throughout — yields an elbow that is
less meaningful than the term suggests, and the sensitivity analysis in Section 5.3 is the
check against over-trusting any single derived value.</br>

2. **A single global `min_samples` per network.** <br>The dimensionality-based anchor treats
every station in a country identically, but does not adapt to local variation in density
within that country's own feature space the way a locally adaptive method might. This is the
standard DBSCAN trade-off, accepted here for its interpretability and its established
literature basis.</br>

3. **No formal feature attribution.** <br>As with Isolation Forest, DBSCAN's cluster
assignment does not report which feature drove any individual station's exclusion from a
cluster. Section 5.2 compares distributions between flagged and consistent stations, which
describes the flagged population but does not attribute an individual verdict to a specific
feature.</br>

4. **Countries with too few stations cannot be assessed.** <br>A network with fewer profiled
stations than `min_samples` cannot support a k-distance derivation at all, and is excluded
from this method entirely rather than assessed on a degenerate basis. Such a network is
carried forward as unassessed by this method, consistent with the pipeline's general
treatment of insufficient data.</br>

5. **Shared dependence on the feature table's construction choices.** <br>Any limitation
attached to the feature table itself — the United States low-value asymmetry, the exclusion
of `pct_very_low` from training, the unequal observation periods across networks — applies
here identically to how it applies in Notebook 04, since both notebooks consume the same
table.</br>

---

### Output

| File | Contents |
|---|---|
| `station_suspicion_dbscan.csv` | Per-station suspicion score, flag, cluster label, and feature profile |

**Next:** `11_autoencoder.ipynb` — a reconstruction-based detector, the ninth and final method before the cross-method consensus.

---

### References

Ester, M., Kriegel, H.-P., Sander, J., & Xu, X. (1996). A density-based algorithm for
discovering clusters in large spatial databases with noise. *Proceedings of the Second
International Conference on Knowledge Discovery and Data Mining (KDD-96)*.

Sander, J., Ester, M., Kriegel, H.-P., & Xu, X. (1998). Density-based clustering in spatial
databases: The algorithm GDBSCAN and its applications. *Data Mining and Knowledge Discovery*,
2(2), 169–194.
